In [24]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

from agno.knowledge.knowledge import Knowledge          # import esplicito dal sottomodulo (più robusto tra versioni)
from agno.vectordb.chroma import ChromaDb
from agno.vectordb.search import SearchType
from agno.knowledge.embedder.google import GeminiEmbedder

load_dotenv()

# --- Setup Vector DB (ChromaDB + embedding Gemini) ---------------------------------
knowledge_base = Knowledge(
    name="CinemaKB",
    vector_db=ChromaDb(
        collection="trame_e_bio",
        path="tmp/chromadb",
        persistent_client=True,
        search_type=SearchType.hybrid,        # combina ricerca vettoriale + full-text (utile per query miste tipo "film di Nolan con astronauti")
        embedder=GeminiEmbedder(id="gemini-embedding-001"),
    ),
)

# --- Setup Graph DB (Neo4j) ---------------------------------------------------------
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")  # niente default hardcoded: se manca, fallisce esplicitamente

if not NEO4J_PASSWORD:
    raise ValueError("NEO4J_PASSWORD non impostata: aggiungila al file .env (evita di scriverla nel codice).")
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise ValueError("GOOGLE_API_KEY / GEMINI_API_KEY non impostata: necessaria per GeminiEmbedder.")

driver_neo4j = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

try:
    driver_neo4j.verify_connectivity()
    print("✅ Connessione a Neo4j stabilita con successo!")
except Exception as e:
    print(f"❌ Errore di connessione a Neo4j: {e}")
    raise


✅ Connessione a Neo4j stabilita con successo!


In [22]:
# Percorsi candidati: adatta questa lista alla struttura reale del tuo progetto.
CANDIDATE_DIRS = [Path("../data"), Path("./data"), Path(".")]

def find_csv(filename: str) -> Path:
    for d in CANDIDATE_DIRS:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(
        f"'{filename}' non trovato in {[str(d) for d in CANDIDATE_DIRS]}. "
        "Aggiorna CANDIDATE_DIRS con il percorso corretto."
    )

movies_df = pd.read_csv(find_csv("tmdb_5000_movies.csv"))
credits_df = pd.read_csv(find_csv("tmdb_5000_credits.csv"))

df = movies_df.merge(credits_df, left_on="id", right_on="movie_id")  # suffissi default: 'title' -> 'title_x' (movies) / 'title_y' (credits)
df = df.drop_duplicates(subset="id").reset_index(drop=True)  # il dataset TMDB contiene alcuni id duplicati

def safe_json_loads(value):
    """json.loads robusto: NaN, stringhe vuote o malformate -> lista vuota invece di eccezione non gestita."""
    if pd.isna(value):
        return []
    try:
        return json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return []

def extract_top_actors(cast_str, top_n=3):
    return [a["name"] for a in safe_json_loads(cast_str)[:top_n] if "name" in a]

def extract_director(crew_str):
    for member in safe_json_loads(crew_str):
        if member.get("job") == "Director":
            return member.get("name", "Sconosciuto")
    return "Sconosciuto"

def extract_genres(genres_str):
    return [g["name"] for g in safe_json_loads(genres_str) if "name" in g]

df["extracted_actors"] = df["cast"].apply(extract_top_actors)
df["extracted_director"] = df["crew"].apply(extract_director)
df["extracted_genres"] = df["genres"].apply(extract_genres)

# Anno di uscita robusto: release_date può essere NaN o in formato inatteso
df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.fillna(0).astype(int)

# Trama pulita e filtro dei record inutilizzabili per il vector store
df["overview_clean"] = df["overview"].fillna("").str.strip()
df = df[df["overview_clean"] != ""].reset_index(drop=True)

print(f"Dataset pronto: {len(df)} film validi (su {len(movies_df)} totali nel CSV originale)")
df[["title_x", "extracted_director", "extracted_actors", "extracted_genres", "release_year"]].head(3)


Dataset pronto: 4799 film validi (su 4803 totali nel CSV originale)


,title_x,extracted_director,extracted_actors,extracted_genres,release_year
0,Avatar,James Cameron,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[Action, Adventure, Fantasy, Science Fiction]",2009
1,Pirates of the Caribbean: At World's End,Gore Verbinski,"[Johnny Depp, Orlando Bloom, Keira Knightley]","[Adventure, Fantasy, Action]",2007
2,Spectre,Sam Mendes,"[Daniel Craig, Christoph Waltz, Léa Seydoux]","[Action, Adventure, Crime]",2015


In [25]:
SCHEMA_QUERIES = [
    "CREATE CONSTRAINT film_vector_id IF NOT EXISTS FOR (f:Film) REQUIRE f.vector_id IS UNIQUE",
    "CREATE CONSTRAINT director_nome  IF NOT EXISTS FOR (d:Director) REQUIRE d.nome IS UNIQUE",
    "CREATE CONSTRAINT actor_nome     IF NOT EXISTS FOR (a:Actor) REQUIRE a.nome IS UNIQUE",
    "CREATE CONSTRAINT genre_nome     IF NOT EXISTS FOR (g:Genre) REQUIRE g.nome IS UNIQUE",
]

with driver_neo4j.session() as session:
    for q in SCHEMA_QUERIES:
        session.run(q)

print("✅ Vincoli di unicità creati (o già presenti).")


✅ Vincoli di unicità creati (o già presenti).


In [26]:
BATCH_CYPHER = """
UNWIND $rows AS row
MERGE (f:Film {vector_id: row.vector_id})
SET f.titolo = row.titolo, f.anno = row.anno
MERGE (d:Director {nome: row.regista})
MERGE (d)-[:DIRECTED]->(f)
WITH f, row
UNWIND row.generi AS genere_nome
MERGE (g:Genre {nome: genere_nome})
MERGE (f)-[:HAS_GENRE]->(g)
WITH f, row
UNWIND row.attori AS attore_nome
MERGE (a:Actor {nome: attore_nome})
MERGE (a)-[:ACTED_IN]->(f)
"""

def salva_batch_nel_grafo(session, rows: list[dict]):
    """Scrive un intero batch di film in un'unica transazione UNWIND."""
    session.run(BATCH_CYPHER, rows=rows)


In [27]:
from time import sleep

N_MOVIES = 50      # Metti None per caricare l'intero dataset
BATCH_SIZE = 25    # righe per transazione Neo4j
CHROMA_MAX_RETRIES = 3

df_sample = df.head(N_MOVIES).copy() if N_MOVIES else df.copy()
print(f"Inizio caricamento di {len(df_sample)} film nel sistema ibrido...")

success_neo4j = 0
success_chroma = 0
errors = []  # (titolo, sistema, messaggio)

def insert_in_chroma(vector_id, titolo, trama, metadata, max_retries=CHROMA_MAX_RETRIES):
    """Inserisce un documento nella Knowledge base (ChromaDB) con l'API ufficiale di Agno."""
    for attempt in range(1, max_retries + 1):
        try:
            knowledge_base.add_content(
                name=f"{titolo} ({vector_id})",
                text_content=trama,
                metadata=metadata,
            )
            return True
        except Exception as e:
            if attempt == max_retries:
                errors.append((titolo, "chroma", str(e)))
                return False
            sleep(2 * attempt)  # backoff esponenziale semplice, utile contro i rate limit di Gemini
    return False

neo4j_batch = []

with driver_neo4j.session() as session:
    for i, (_, row) in enumerate(df_sample.iterrows(), start=1):
        titolo = row["title_x"]
        anno = int(row["release_year"])
        trama = row["overview_clean"]
        regista = row["extracted_director"]
        attori = row["extracted_actors"] or []
        generi = row["extracted_genres"] or []
        vector_id = f"movie_{row['id']}"

        # --- 1. ChromaDB (vettori) ---
        meta = {"tipo": "trama", "vector_id": vector_id, "titolo": titolo, "anno": anno}
        if insert_in_chroma(vector_id, titolo, trama, meta):
            success_chroma += 1

        # --- 2. Neo4j (accumula per il batch) ---
        neo4j_batch.append({
            "vector_id": vector_id, "titolo": titolo, "anno": anno,
            "regista": regista, "attori": attori, "generi": generi,
        })

        if len(neo4j_batch) >= BATCH_SIZE or i == len(df_sample):
            try:
                salva_batch_nel_grafo(session, neo4j_batch)
                success_neo4j += len(neo4j_batch)
            except Exception as e:
                errors.append((f"batch terminato a '{titolo}'", "neo4j", str(e)))
            neo4j_batch = []

        if i % 10 == 0 or i == len(df_sample):
            print(f"Processati {i}/{len(df_sample)} — Chroma: {success_chroma} · Neo4j: {success_neo4j}")

print("\n--- Riepilogo ---")
print(f"✅ Chroma: {success_chroma}/{len(df_sample)}")
print(f"✅ Neo4j:  {success_neo4j}/{len(df_sample)}")
if errors:
    print(f"⚠️  {len(errors)} errori totali (primi 5):")
    for titolo, sistema, msg in errors[:5]:
        print(f"  [{sistema}] {titolo}: {msg}")


Inizio caricamento di 50 film nel sistema ibrido...


INFO Adding content from Avatar (movie_19995)

INFO Selecting reader for extension: Text

INFO Upserting 1 documents

ERROR    GOOGLE_API_KEY not set. Please set the GOOGLE_API_KEY environment variable.

...

INFO Selecting reader for extension: Text

INFO Upserting 1 documents

ERROR    Error upserting documents by content hash                                                                 
         Traceback (most recent call last):                                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 495, in upsert       
             self._upsert(content_hash, documents, filters)                                                        
             ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 557, in _upsert      
             self._batch_operation(                                                                                
             ~~~~~~~~~~~~~~~~~~~~~^                                                                                
                 ids=ids,                                                                                          
                 ^^^^^^^^                                                                                          
             ...<4 lines>...                                                                                       
                 operation_func=self._collection.upsert,                                                           
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                           
             )                                                                                                     
             ^                                                                                                     
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 255, in              
         _batch_operation                                                                                          
             operation_func(                                                                                       
             ~~~~~~~~~~~~~~^                                                                                       
                 ids=batch_ids,                                                                                    
                 ^^^^^^^^^^^^^^                                                                                    
             ...<2 lines>...                                                                                       
                 metadatas=batch_metadata,                                                                         
                 ^^^^^^^^^^^^^^^^^^^^^^^^^                                                                         
             )                                                                                                     
             ^                                                                                                     
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/chromadb/api/models/Collection.py", line 530, in upsert      
             self._client._upsert(                                                                                 
             ~~~~~~~~~~~~~~~~~~~~^                                                                                 
                 collection_id=self.id,                                                                            
                 ^^^^^^^^^^^^^^^^^^^^^^                 

ERROR    Error upserting document: Collection expecting embedding with dimension of 384, got 1536

Processati 50/50 — Chroma: 50 · Neo4j: 50

--- Riepilogo ---
✅ Chroma: 50/50
✅ Neo4j:  50/50


In [ ]:
# Conteggio nodi nel grafo
with driver_neo4j.session() as session:
    counts = session.run("""
        MATCH (f:Film)     WITH count(f) AS film
        MATCH (d:Director) WITH film, count(d) AS registi
        MATCH (a:Actor)    WITH film, registi, count(a) AS attori
        MATCH (g:Genre)    RETURN film, registi, attori, count(g) AS generi
    """).single()
    print("Neo4j ->", dict(counts))

# Ricerca semantica di prova in ChromaDB (via Agno)
risultati = knowledge_base.search("un supereroe che deve salvare la città", num_documents=3)
print("\nChromaDB -> risultati per 'un supereroe che deve salvare la città':")
for r in risultati:
    anteprima = (r.content or "")[:120].replace("\n", " ")
    print(f"- {r.name}: {anteprima}...")

# Quando hai finito la sessione di lavoro:
# driver_neo4j.close()
